# Diagnóstico de cancro da mama — Passive Learning (baseline)

**Inteligência Artificial Centrada no Humano** · Mestrado em Inteligência Artificial e Engenharia de Dados · FCTUC

Treina a CNN com **todo** o conjunto de treino anotado. Os resultados servem de referência para as experiências de Active Learning (`02_active_learning.ipynb`), que usam muito menos imagens anotadas.

> **Estado**
> - ✅ Feito: código completo e testado com uma configuração reduzida; métricas de recall, F1 e falsos negativos.
> - ⬜ Falta: correr o treino completo (ainda não há resultados) e escrever as conclusões.

**Índice**
1. Configuração
2. Dados
3. Treino
4. Avaliação
5. Conclusões

## 1. Configuração

A primeira célula prepara o ambiente e funciona tanto no **Google Colab** como **localmente**:
- no Colab, monta o Google Drive (onde deve estar uma cópia do repositório com o dataset) e instala as bibliotecas que faltam (`modAL-python`, `scikeras`);
- localmente, assume que o notebook foi aberto a partir da pasta `notebooks/`.

Depois junta a raiz do repositório ao `sys.path`, para se poder importar o código partilhado da pasta `src/`. Os caminhos do dataset e da pasta de resultados estão em `src/config.py`.

In [ ]:
# Ambiente: monta o Google Drive no Colab; localmente usa a pasta do repositório
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/miacd-iach-breast-cancer-diagnosis")  # pasta do repositório no Drive
    %pip install -q modAL-python scikeras
else:
    ROOT = Path.cwd().parent

sys.path.insert(0, str(ROOT))
from src.config import DATA_PATH, RESULTS_PATH

In [ ]:
# Parâmetros
IMG_SIZE = 224          # reduzir (ex.: 128) se faltar memória
EPOCHS = 15

## 2. Dados

`load_dataset` (em `src/data.py`) faz cinco passos:

1. **Ler** as imagens em escala de cinzento e redimensionar para `IMG_SIZE × IMG_SIZE`. Os valores dos píxeis passam de 0–255 para 0–1 (normalização), o que ajuda o treino a convergir.
2. **Tirar a classe do nome do ficheiro** (ex.: `benign (12).png` → `benign`). As labels numéricas são `0 = normal`, `1 = benign`, `2 = malignant`. As imagens duplicadas são excluídas: o dataset tem a mesma imagem como `benign (433)` e `malignant (145)`, e não se sabe qual das labels está certa. Ficam 778 imagens.
3. **Dividir em três conjuntos**, de forma estratificada (cada um fica com as mesmas proporções de classes):
   - **treino (65%)**: as imagens com que o modelo aprende;
   - **validação (15%)**: para acompanhar o treino e escolher parâmetros (épocas, batch size…);
   - **teste (20%)**: só para a avaliação final.
4. **Balancear só o treino** com rotações de 90º e reflexões, até cada classe ter um número parecido de imagens. A validação e o teste ficam com as imagens originais, para medir o desempenho em condições reais.
5. **Baralhar o treino**, para as classes ficarem misturadas.

> **Porque é que a divisão vem antes do balanceamento?** Se fosse ao contrário, uma imagem podia ir para o treino e a sua versão rodada para a validação ou o teste. O modelo seria avaliado com imagens que, na prática, já viu — isto chama-se *data leakage* e dá resultados otimistas e falsos. Assim, **nenhuma imagem de validação ou de teste, nem uma cópia dela, é usada no treino.**

In [ ]:
# Carregar, dividir em treino/validação/teste e balancear só o treino
import numpy as np
from src.data import CLASSES, load_dataset, summarize

x_train, x_val, x_test, y_train, y_val, y_test = load_dataset(DATA_PATH, size=IMG_SIZE)
summarize(x_train, x_val, x_test, y_train, y_val, y_test)

## 3. Treino

### O que é uma CNN?
Uma **Rede Neuronal Convolucional** aprende sozinha que padrões da imagem (contornos, texturas, formas) ajudam a distinguir as classes. As primeiras camadas detetam padrões simples, como arestas; as seguintes combinam-nos em padrões mais complexos, como a forma de uma lesão.

### A nossa arquitetura (`src/model.py`)
- **4 blocos convolucionais** com 32 → 64 → 128 → 32 filtros de 4×4. Cada bloco tem:
  - `Conv2D` + ReLU: aplica os filtros e deteta padrões;
  - `BatchNormalization`: normaliza as ativações, o que estabiliza o treino;
  - `MaxPooling2D`: reduz a imagem para metade, guardando só os valores mais fortes;
  - `Dropout(0.3)` nos blocos 2 e 3: desliga 30% dos neurónios ao acaso em cada passo, para o modelo não decorar o treino (*overfitting*).
- **`Flatten` + `Dense(32)`**: junta tudo num vetor e combina os padrões encontrados.
- **`Dense(3, softmax)`**: devolve 3 probabilidades que somam 1, uma por classe. A classe prevista é a de maior probabilidade.

### Como é treinada
- **Labels em one-hot:** a classe 2 passa a `[0, 0, 1]`, para comparar com as 3 probabilidades da saída.
- **Função de perda:** `categorical_crossentropy`, que penaliza mais quando o modelo dá pouca probabilidade à classe certa.
- **Otimizador:** Adam com learning rate 0.00001 (baixo: aprende devagar, mas de forma estável).
- **Épocas (`EPOCHS`):** quantas vezes o modelo vê o conjunto de treino inteiro.
- **Validação:** em cada época, o modelo é avaliado no conjunto de validação (imagens que nunca usa para aprender), para ver se está a generalizar ou só a decorar. Se a accuracy de treino sobe e a de validação não, é *overfitting*.

In [ ]:
from tensorflow.keras.utils import to_categorical
from src.model import build_cnn

y_train_one_hot = to_categorical(y_train, num_classes=len(CLASSES))
y_val_one_hot = to_categorical(y_val, num_classes=len(CLASSES))

model = build_cnn(size=IMG_SIZE)
history = model.fit(x_train, y_train_one_hot, epochs=EPOCHS, validation_data=(x_val, y_val_one_hot), verbose=1)

## 4. Avaliação

O modelo é avaliado no **conjunto de teste**, que nunca viu durante o treino.

| Métrica | O que mede |
|---|---|
| **Accuracy** | % de imagens bem classificadas. Pode enganar quando as classes não estão equilibradas. |
| **Recall** (por classe) | Dos casos que são mesmo dessa classe, quantos o modelo apanhou. `recall_macro` é a média das 3 classes. |
| **F1** | Equilíbrio entre recall e precisão (dos casos que o modelo diz serem dessa classe, quantos são mesmo). `f1_macro` é a média das 3 classes. |
| **Recall de maligno** | Dos cancros reais, quantos foram detetados. **É a métrica mais importante do projeto.** |
| **Falsos negativos (FN)** | Nº de cancros classificados como benignos ou normais. |

> **Porque é que o foco são os falsos negativos?** Num diagnóstico, um falso positivo (dizer que é maligno quando não é) leva a mais exames, o que é incómodo mas não grave. Um falso negativo (não detetar um cancro) pode atrasar o tratamento e pôr a vida em risco. Por isso preferimos um modelo com menos falsos negativos, mesmo que a accuracy seja um pouco menor.

As métricas ficam guardadas em `results/passive_learning.json`, para o notebook 02 as usar como baseline.

In [ ]:
from src.evaluation import compute_metrics, save_json

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
metrics = compute_metrics(y_test, y_pred)
metrics["accuracy_train"] = float(model.evaluate(x_train, y_train_one_hot, verbose=0)[1])
metrics["accuracy_test"] = metrics["accuracy"]

save_json(metrics, RESULTS_PATH / "passive_learning.json")
metrics

### Como ler a matriz de confusão
- Cada **linha** é a classe real e cada **coluna** é a classe prevista.
- A **diagonal** são os acertos.
- Os **falsos negativos** estão na linha `malignant`, nas colunas `normal` e `benign`.

In [ ]:
from src.evaluation import plot_confusion_matrix

plot_confusion_matrix(y_test, y_pred, title="Passive Learning")

## 5. Conclusões

*A preencher depois de correr o treino completo.*